# Section 2: Low NA Circular Aperture (ε = 0)

**System:** NA = 0.1, λ = 532 nm, circular aperture (no obscuration), medium: air (n = 1).

At low numerical aperture (NA ≪ 1) the paraxial approximation is valid and scalar diffraction theory predicts nearly identical results to the full vectorial Richards-Wolf treatment. The longitudinal field component Ez is negligibly small.

**Goals:**
- Compute focal-plane radial intensity for uniform and Gaussian (α = 1, 2, 4) inputs
- Compute axial intensity I(r=0, z) — depth of focus
- Compare RichardsWolfSimulator with FocusedGaussianBeamTheory (Tanaka et al.)
- Show field components Ex, Ey, Ez for x-polarized input

> **Note on computation time:** The Richards-Wolf integrals are computed point-by-point using numerical quadrature. We use small arrays (50–80 points) for speed. Higher resolution takes proportionally longer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import sys
sys.path.insert(0, '/Users/raaromero/Projects/Research/optical-diffraction/src')

from optical_diffraction import RichardsWolfSimulator, FocusedGaussianBeamTheory

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.dpi': 100})

# System parameters
NA = 0.1
wavelength = 0.532   # microns (532 nm)
n_medium = 1.0
epsilon = 0.0        # circular aperture

# Airy radius for reference
airy_radius = 0.61 * wavelength / NA
print(f"System: NA={NA}, λ={wavelength} μm, n={n_medium}")
print(f"Airy radius: {airy_radius:.3f} μm")
print(f"Depth of focus (±λ/NA²): ±{wavelength/NA**2:.1f} μm")

## 2.1 Focal Plane Radial Intensity — Uniform and Gaussian Inputs

Radial intensity profiles I(r, z=0) at the focal plane for different input field types.

In [ ]:
# Radial coordinate at focal plane (in microns)
# Extend to ~4 Airy radii to capture side lobes
r_max = 4.5 * airy_radius
r = np.linspace(0, r_max, 80)
z_focal = np.zeros_like(r)  # z=0 is the focal plane

# Define input field configurations
configs = [
    ('uniform', None, 'Uniform', 'tab:blue', '-'),
    ('gaussian', 1, 'Gaussian α=1', 'tab:orange', '-'),
    ('gaussian', 2, 'Gaussian α=2', 'tab:green', '-'),
    ('gaussian', 4, 'Gaussian α=4', 'tab:red', '-'),
]

fig, ax = plt.subplots(figsize=(9, 5))

intensities_focal = {}
for (field_type, alpha, label, color, ls) in configs:
    trunc = alpha if alpha is not None else 0.0
    sim = RichardsWolfSimulator(
        wavelength=wavelength,
        numerical_aperture=NA,
        n_medium=n_medium,
        input_field=field_type,
        truncation_coeff=trunc,
    )
    Ex, Ey, Ez = sim.compute_field(r, z_focal)
    I = np.abs(Ex)**2 + np.abs(Ey)**2 + np.abs(Ez)**2
    I_norm = I / I.max()
    intensities_focal[label] = I_norm
    ax.plot(r / airy_radius, I_norm, color=color, ls=ls, lw=2, label=label)

ax.axvline(x=1.0, color='black', ls=':', lw=1.2, alpha=0.6, label='Airy radius')
ax.set_xlabel('Radial distance r / r$_{Airy}$')
ax.set_ylabel('Normalized intensity I/I$_{max}$')
ax.set_title(f'Focal Plane Radial Intensity — Low NA (NA={NA}, λ={wavelength} μm)')
ax.set_xlim(0, r_max / airy_radius)
ax.set_ylim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.show()

print("\nNote: Uniform illumination gives the narrowest Airy pattern. Gaussian inputs")
print("broaden the central peak but reduce side-lobe levels.")

## 2.2 Axial Intensity Profile I(r=0, z)

The on-axis intensity shows the depth of focus. For low NA, the paraxial result gives a sinc²-like profile for uniform illumination.

In [ ]:
# Axial coordinate — sample symmetrically around focus
dof = wavelength / NA**2  # approximate depth of focus
z_max = 3.0 * dof
z_axial = np.linspace(-z_max, z_max, 60)
r_zero = np.zeros_like(z_axial)  # on-axis (r=0)

fig, ax = plt.subplots(figsize=(10, 5))

intensities_axial = {}
for (field_type, alpha, label, color, ls) in configs:
    trunc = alpha if alpha is not None else 0.0
    sim = RichardsWolfSimulator(
        wavelength=wavelength,
        numerical_aperture=NA,
        n_medium=n_medium,
        input_field=field_type,
        truncation_coeff=trunc,
    )
    Ex, Ey, Ez = sim.compute_field(r_zero, z_axial)
    I = np.abs(Ex)**2 + np.abs(Ey)**2 + np.abs(Ez)**2
    I_norm = I / I.max()
    intensities_axial[label] = I_norm
    ax.plot(z_axial / dof, I_norm, color=color, ls=ls, lw=2, label=label)

ax.axvline(x=0, color='black', ls='-', lw=1, alpha=0.4)
ax.axhline(y=0.5, color='black', ls=':', lw=1, alpha=0.5, label='FWHM level')
ax.set_xlabel('Axial position z / DoF  (DoF = λ/NA²)')
ax.set_ylabel('Normalized on-axis intensity I(r=0, z)/I$_{max}$')
ax.set_title(f'Axial Intensity Profile — Low NA (NA={NA})')
ax.legend()
plt.tight_layout()
plt.show()

print(f"DoF reference: λ/NA² = {dof:.2f} μm")

## 2.3 Comparison: RichardsWolfSimulator vs FocusedGaussianBeamTheory

At low NA the Richards-Wolf vectorial result should agree closely with the scalar Gaussian beam theory (Tanaka et al. 1985). We compare both focal-plane and axial profiles for α = 1, 2, 4.

In [ ]:
alphas_compare = [1, 2, 4]
colors_compare = ['tab:orange', 'tab:green', 'tab:red']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Focal plane comparison ---
ax = axes[0]
for alpha, color in zip(alphas_compare, colors_compare):
    # Richards-Wolf (vectorial)
    sim_rw = RichardsWolfSimulator(
        wavelength=wavelength, numerical_aperture=NA, n_medium=n_medium,
        input_field='gaussian', truncation_coeff=alpha,
    )
    Ex, Ey, Ez = sim_rw.compute_field(r, z_focal)
    I_rw = np.abs(Ex)**2 + np.abs(Ey)**2 + np.abs(Ez)**2
    I_rw = I_rw / I_rw.max()
    ax.plot(r / airy_radius, I_rw, color=color, lw=2.5, label=f'RW α={alpha}')

    # Tanaka analytical theory
    theory = FocusedGaussianBeamTheory(
        numerical_aperture=NA, wavelength=wavelength, n_medium=n_medium,
        truncation_coeff=alpha,
    )
    I_theory = theory.focal_plane_intensity(r)
    ax.plot(r / airy_radius, I_theory, color=color, lw=1.5, ls='--',
            label=f'Tanaka α={alpha}')

ax.axvline(x=1.0, color='black', ls=':', lw=1.2, alpha=0.5)
ax.set_xlabel('r / r$_{Airy}$')
ax.set_ylabel('Normalized intensity')
ax.set_title('Focal Plane: RW (solid) vs Tanaka (dashed)')
ax.set_xlim(0, r_max / airy_radius)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)

# --- Axial comparison ---
ax = axes[1]
for alpha, color in zip(alphas_compare, colors_compare):
    # Richards-Wolf axial
    sim_rw = RichardsWolfSimulator(
        wavelength=wavelength, numerical_aperture=NA, n_medium=n_medium,
        input_field='gaussian', truncation_coeff=alpha,
    )
    Ex, Ey, Ez = sim_rw.compute_field(r_zero, z_axial)
    I_rw = np.abs(Ex)**2 + np.abs(Ey)**2 + np.abs(Ez)**2
    I_rw = I_rw / I_rw.max()
    ax.plot(z_axial / dof, I_rw, color=color, lw=2.5, label=f'RW α={alpha}')

    # Tanaka axial
    theory = FocusedGaussianBeamTheory(
        numerical_aperture=NA, wavelength=wavelength, n_medium=n_medium,
        truncation_coeff=alpha,
    )
    # axial_intensity uses absolute z coordinates; z_focus=0 by default
    I_theory_ax = theory.axial_intensity(z_axial)
    if hasattr(I_theory_ax, '__len__') and np.max(I_theory_ax) > 0:
        I_theory_ax = I_theory_ax / np.max(I_theory_ax)
    ax.plot(z_axial / dof, I_theory_ax, color=color, lw=1.5, ls='--',
            label=f'Tanaka α={alpha}')

ax.axvline(x=0, color='black', ls='-', lw=0.8, alpha=0.4)
ax.set_xlabel('z / DoF')
ax.set_ylabel('Normalized on-axis intensity')
ax.set_title('Axial Profile: RW (solid) vs Tanaka (dashed)')
ax.legend(fontsize=9)

fig.suptitle(f'Low NA (NA={NA}) Comparison: Richards-Wolf vs Tanaka Theory', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 2.4 Field Components Ex, Ey, Ez for x-Polarized Input

For x-polarized input at low NA, the field is dominated by Ex. The longitudinal component Ez and the cross-polarized Ey are expected to be negligibly small at NA = 0.1.

In [ ]:
# Use uniform illumination to show the clearest Airy-like result
sim_uniform = RichardsWolfSimulator(
    wavelength=wavelength,
    numerical_aperture=NA,
    n_medium=n_medium,
    polarization='x',
    input_field='uniform',
    truncation_coeff=0.0,
)

Ex, Ey, Ez = sim_uniform.compute_field(r, z_focal)
Ix = np.abs(Ex)**2
Iy = np.abs(Ey)**2
Iz = np.abs(Ez)**2
I_total = Ix + Iy + Iz
I_total_norm = I_total / I_total.max()
scale = I_total.max()  # normalize all to same scale

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Absolute (normalized to total)
ax = axes[0]
ax.plot(r / airy_radius, Ix / scale, lw=2.5, color='tab:blue', label='|Ex|² (x-component)')
ax.plot(r / airy_radius, Iy / scale, lw=2, color='tab:orange', ls='--', label='|Ey|² (cross-pol)')
ax.plot(r / airy_radius, Iz / scale, lw=2, color='tab:red', ls=':', label='|Ez|² (longitudinal)')
ax.plot(r / airy_radius, I_total_norm, lw=1.5, color='black', ls='-', alpha=0.5, label='Total')
ax.axvline(x=1.0, color='black', ls=':', lw=1.0, alpha=0.5)
ax.set_xlabel('r / r$_{Airy}$')
ax.set_ylabel('Intensity (normalized to total peak)')
ax.set_title('Field Components at Focal Plane (x-polarization, uniform)')
ax.legend()

# Log scale to see weak components
ax = axes[1]
ax.semilogy(r / airy_radius, np.maximum(Ix / scale, 1e-10), lw=2.5, color='tab:blue', label='|Ex|²')
ax.semilogy(r / airy_radius, np.maximum(Iy / scale, 1e-10), lw=2, color='tab:orange', ls='--', label='|Ey|²')
ax.semilogy(r / airy_radius, np.maximum(Iz / scale, 1e-10), lw=2, color='tab:red', ls=':', label='|Ez|²')
ax.axvline(x=1.0, color='black', ls=':', lw=1.0, alpha=0.5)
ax.set_xlabel('r / r$_{Airy}$')
ax.set_ylabel('Intensity (log scale)')
ax.set_title('Field Components — Log Scale (Vectorial Effects)')
ax.set_ylim(1e-10, 2)
ax.legend()

fig.suptitle(f'Field Components at Low NA (NA={NA}) — x-Polarized Uniform Input', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# Peak ratios
print("Peak intensity ratios (relative to total):")
print(f"  |Ex|²_max / I_total_max = {Ix.max()/scale:.6f}")
print(f"  |Ey|²_max / I_total_max = {Iy.max()/scale:.2e}  (negligible at low NA)")
print(f"  |Ez|²_max / I_total_max = {Iz.max()/scale:.2e}  (negligible at low NA)")
print(f"\nAt NA={NA}, longitudinal component Ez is ~(NA/2)² ≈ {(NA/2)**2:.4f} relative to Ex.")

## 2.5 Field Components for Gaussian Inputs

Showing components Ex, Ez for Gaussian inputs (α=1,2,4) on the focal plane.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_a = ['tab:orange', 'tab:green', 'tab:red']

ax_ex, ax_ez = axes

for alpha, color in zip([1, 2, 4], colors_a):
    sim_g = RichardsWolfSimulator(
        wavelength=wavelength, numerical_aperture=NA, n_medium=n_medium,
        polarization='x', input_field='gaussian', truncation_coeff=alpha,
    )
    Ex, Ey, Ez = sim_g.compute_field(r, z_focal)
    I_total = np.abs(Ex)**2 + np.abs(Ey)**2 + np.abs(Ez)**2
    peak = I_total.max()
    ax_ex.plot(r / airy_radius, np.abs(Ex)**2 / peak, color=color, lw=2, label=f'α={alpha}')
    ax_ez.plot(r / airy_radius, np.abs(Ez)**2 / peak, color=color, lw=2, label=f'α={alpha}')

for ax, title_sfx, ylim_top in [(ax_ex, '|Ex|²', 1.05), (ax_ez, '|Ez|²', None)]:
    ax.set_xlabel('r / r$_{Airy}$')
    ax.set_ylabel(f'{title_sfx} (normalized to total peak)')
    ax.set_title(f'Focal Plane {title_sfx} — Low NA (NA={NA}), Gaussian inputs')
    ax.axvline(x=1.0, color='black', ls=':', lw=1.0, alpha=0.5)
    ax.legend()
    if ylim_top:
        ax.set_ylim(0, ylim_top)

plt.tight_layout()
plt.show()

print("\nConclusion: At low NA, Ez is negligible for all input profiles.")
print("Scalar diffraction theory is an excellent approximation.")

## Summary

| Observation | Details |
|---|---|
| Focal spot width | Larger α (more Gaussian-like) → slightly broader central peak, reduced side lobes |
| RW vs Tanaka agreement | Excellent at low NA — vectorial and scalar theories agree |
| Ez/Ex ratio | ~(NA/2)² ≈ 0.0025 at NA=0.1 — completely negligible |
| Depth of focus | All inputs give similar axial extent; more Gaussian inputs → slightly broader axial profile |

The next notebook extends this analysis to annular apertures at low NA.